In [1]:
def read_dgsp_input(filename):
    with open(filename, "r") as file:
        lines = [line.strip() for line in file if line.strip()]

    # Line 1: vertex names without colors
    vertex_names = lines[0].split()
    n = len(vertex_names)

    # Next n lines: directed adjacency matrix
    adj_matrix = []
    for i in range(1, n + 1):
        row = list(map(int, lines[i].split()))
        adj_matrix.append(row)

    # Next line: u v
    u, v = lines[n + 1].split()

    # Last line: l
    l = int(lines[n + 2])

    return vertex_names, adj_matrix, u, v, l


def transform_dgsp_to_3csp(vertex_names, adj_matrix, u, v, l):
    """
    Converts DGSP into 3CSP.

    Idea:
    Each original vertex x becomes three colored copies:
    xb, xw, xr

    A directed edge x -> y becomes an undirected edge from xr to yb.

    This forces paths to move in the cycle:
    b -> w -> r -> b
    """

    new_vertices = []
    new_colors = {}

    # Create 3 colored copies for every original vertex
    for vertex in vertex_names:
        for color in ["b", "w", "r"]:
            new_name = vertex + color
            new_vertices.append(new_name)
            new_colors[new_name] = color

    # Create empty adjacency matrix
    n_new = len(new_vertices)
    new_matrix = []

    for i in range(n_new):
        row = []
        for j in range(n_new):
            row.append(0)
        new_matrix.append(row)

    # Helper dictionary: vertex name -> index
    index_map = {}
    for i in range(n_new):
        index_map[new_vertices[i]] = i

    # Add internal edges for each vertex:
    # xb -- xw -- xr
    for vertex in vertex_names:
        xb = vertex + "b"
        xw = vertex + "w"
        xr = vertex + "r"

        xb_index = index_map[xb]
        xw_index = index_map[xw]
        xr_index = index_map[xr]

        new_matrix[xb_index][xw_index] = 1
        new_matrix[xw_index][xb_index] = 1

        new_matrix[xw_index][xr_index] = 1
        new_matrix[xr_index][xw_index] = 1

    # Add edges for every directed edge x -> y
    original_n = len(vertex_names)

    for i in range(original_n):
        for j in range(original_n):
            if adj_matrix[i][j] == 1:
                x = vertex_names[i]
                y = vertex_names[j]

                from_vertex = x + "r"
                to_vertex = y + "b"

                from_index = index_map[from_vertex]
                to_index = index_map[to_vertex]

                # 3CSP graph is undirected
                new_matrix[from_index][to_index] = 1
                new_matrix[to_index][from_index] = 1

    # Start at u_b and end at v_b
    new_s = u + "b"
    new_t = v + "b"

    # Each directed edge becomes 3 steps:
    # xb -> xw -> xr -> yb
    new_k = 3 * l

    return new_vertices, new_colors, new_matrix, new_s, new_t, new_k


def print_3csp_instance(new_vertices, new_colors, new_matrix, s, t, k):
    # First line: vertices with colors
    first_line = []

    for vertex in new_vertices:
        first_line.append(vertex)

    print(" ".join(first_line))

    # Adjacency matrix
    for row in new_matrix:
        print(" ".join(map(str, row)))

    # s t
    print(s, t)

    # k
    print(k)


def main():
    filename = input("Enter DGSP input file name: ")

    try:
        vertex_names, adj_matrix, u, v, l = read_dgsp_input(filename)

        new_vertices, new_colors, new_matrix, s, t, k = transform_dgsp_to_3csp(
            vertex_names,
            adj_matrix,
            u,
            v,
            l
        )

        print_3csp_instance(new_vertices, new_colors, new_matrix, s, t, k)

    except FileNotFoundError:
        print("Error: File not found.")
    except Exception as e:
        print("Error:", e)


if __name__ == "__main__":
    main()

Enter DGSP input file name:  dagsp_input.txt


1b 1w 1r 2b 2w 2r 3b 3w 3r 4b 4w 4r
0 1 0 0 0 0 0 0 0 0 0 0
1 0 1 0 0 0 0 0 0 0 0 0
0 1 0 1 0 0 1 0 0 0 0 0
0 0 1 0 1 0 0 0 0 0 0 0
0 0 0 1 0 1 0 0 0 0 0 0
0 0 0 0 1 0 0 0 0 1 0 0
0 0 1 0 0 0 0 1 0 0 0 0
0 0 0 0 0 0 1 0 1 0 0 0
0 0 0 0 0 0 0 1 0 1 0 0
0 0 0 0 0 1 0 0 1 0 1 0
0 0 0 0 0 0 0 0 0 1 0 1
0 0 0 0 0 0 0 0 0 0 1 0
1b 4b
6
